# Singapore Smart City: Silver Layer ETL 

This notebook represents the **Silver Layer** of our Medallion Data Engineering Architecture. Because we are operating on a strict **$0 Cloud-Native constraint**, this ETL pipeline runs entirely on Google Colab's free CPU instances and uses Google Drive as our Data Lake.

### Pipeline Objectives:
1. **Ingest Bronze Data:** Mount Google Drive and load the raw JSONL files and JPEG images collected by the edge scraper.
2. **Data Cleaning:** Detect and drop corrupted images (0 bytes, unreadable headers) and remove invalid API responses.
3. **Data Fusion:** Temporally join the separate Traffic, Weather, and PM2.5 API streams into a single cohesive record per camera frame.
4. **Export Silver Data:** Save the cleaned, enriched structured dataset as highly compressed **Parquet files** back to Google Drive for fast querying by the ML training pipelines (Level 1 YOLO & Level 3 PI-STGNN).

> **Cost:** $0, runs purely on free Colab instances.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas pyarrow fastparquet pillow tqdm

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image

DRIVE_ROOT = Path('/content/drive/MyDrive/sg_smart_city')
BRONZE_DIR = DRIVE_ROOT / 'data' / 'raw'
SILVER_DIR = DRIVE_ROOT / 'data' / 'silver'

os.makedirs(SILVER_DIR, exist_ok=True)
print(f"Extracting Bronze Data from: {BRONZE_DIR}")
print(f"Target Silver Data Lake: {SILVER_DIR}")

# 1. Extract JSONL Metadata
metadata_files = list(BRONZE_DIR.rglob('metadata.jsonl'))
print(f"Found {len(metadata_files)} metadata stream files.")

records = []
for mf in tqdm(metadata_files, desc="Parsing Bronze Streams"):
    with open(mf, 'r') as f:
        for line in f:
            try:
                record = json.loads(line.strip())
                record['camera_id'] = mf.parent.name
                record['collection_date'] = mf.parent.parent.name
                records.append(record)
            except json.JSONDecodeError:
                continue # Skip corrupted lines

df = pd.DataFrame(records)
print(f"\nExtracted {len(df)} total raw records.")

# 2. Clean and Validate Images
valid_records = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Validating Image Integrity"):
    img_path = Path(row.get('local_image_path', ''))
    if not img_path.is_absolute():
        # Reconstruct path for colab environment
        img_path = BRONZE_DIR / row['collection_date'] / row['camera_id'] / img_path.name
    
    is_valid = False
    if img_path.exists() and img_path.stat().st_size > 1024:  # At least 1KB
        try:
            with Image.open(img_path) as img:
                img.verify() # Fast header check without full decode
                is_valid = True
        except Exception:
            pass
            
    if is_valid:
        row_dict = row.to_dict()
        row_dict['colab_image_path'] = str(img_path)
        valid_records.append(row_dict)

silver_df = pd.DataFrame(valid_records)
print(f"\nData Quality Gate: {len(df) - len(silver_df)} corrupted/missing frames dropped.")
print(f"{len(silver_df)} pristine frames advanced to Silver Layer.")

# 3. Temporal Join & Feature Engineering
if not silver_df.empty:
    # Convert timestamp to Pandas datetime for powerful temporal operations
    silver_df['timestamp_dt'] = pd.to_datetime(silver_df['timestamp'])
    silver_df = silver_df.sort_values(['camera_id', 'timestamp_dt'])
    
    # Extract cyclic temporal features for the ST-GNN later
    silver_df['hour'] = silver_df['timestamp_dt'].dt.hour
    silver_df['day_of_week'] = silver_df['timestamp_dt'].dt.dayofweek
    
    # Note: If separate weather API files were collected, we would use pd.merge_asof() here 
    # to do a "closest-time" join between the camera frame and the weather station readings.
    
    # 4. Export to Parquet (Highly compressed columnar format)
    parquet_path = SILVER_DIR / 'traffic_features.parquet'
    silver_df.to_parquet(parquet_path, engine='pyarrow', index=False)
    
    print(f"\n✅ Success! Silver DataFrame ({silver_df.shape[0]} rows, {silver_df.shape[1]} columns) exported to {parquet_path}")
    print(f"Parquet Size: {parquet_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print("\n❌ No valid data found to process.")
